### 1. Extract Folder Topic Names

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re

In [2]:
dataset_dir = Path('./20_newsgroups')
topic_names = sorted([topic.name for topic in dataset_dir.iterdir() if topic.is_dir()])
print(f"Number of topics: {len(topic_names)}")

Number of topics: 20


### 2. Data Loading Pipeline

In [3]:
def clean_newsgroup_body(raw_text):
    parts = raw_text.split('\n\n', 1) # headers from body
    body = parts[1] if len(parts) > 1 else raw_text
    text = body.lower()
    text = re.sub(r'^\s*[|>#]{2,}\s*$', '', text, flags=re.MULTILINE) # remove lines starting with >, |, # (common in quoted text)
    text = re.sub(r'(^|\n)\s*>[^\n]*', '', text) # remove quoted text, > & spaces
    text = re.sub(r'[^a-z\s]', ' ', text) # remove non-alphabetic characters, keep spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces and newlines, replace with single space
    text = re.sub(r'\S*@\S*\s?', '', text) # remove email addresses
    text = re.sub(r'\s+', ' ', text).strip() # removes white spaces and newlines, replaces with single space
    return text

In [4]:
def load_dataset_to_dataframe(folder_path):
    data_records = []
    
    for label_dir in os.listdir(folder_path): # iterate through root newsgroup folders
        dir_path = os.path.join(folder_path, label_dir)
        
        if not os.path.isdir(dir_path): # skipping non-directory files if any
            continue
            
        for file_name in os.listdir(dir_path): # iterate through files in each category folder
            file_path = os.path.join(dir_path, file_name)
            try:
                with open(file_path, "r", encoding="latin-1") as f: # read and use latin-1 encoding to prevent UnicodeDecodeErrors
                    raw_content = f.read()
                    cleaned_text = clean_newsgroup_body(raw_content) # clean the text using the defined function
                    if cleaned_text:
                        data_records.append({"file_id": file_name, "label": label_dir, 
                        "text": cleaned_text, "original_length": len(raw_content)}) # append only cleaned text
            except Exception as e:
                print(f"Failed to read {file_path}: {e}")
                
    return pd.DataFrame(data_records) # convert dict list of records to a DataFrame

In [5]:
dataset_path = "./20_newsgroups" 
df = load_dataset_to_dataframe(dataset_path)

print(f"Successfully loaded {len(df)} documents.")
print("\nClass Distribution:")
print(df['label'].value_counts())

Successfully loaded 19955 documents.

Class Distribution:
label
talk.politics.mideast       1000
talk.politics.misc          1000
talk.politics.guns          1000
talk.religion.misc          1000
alt.atheism                  999
rec.sport.baseball           999
rec.sport.hockey             999
sci.crypt                    999
sci.electronics              999
sci.space                    999
rec.autos                    998
sci.med                      998
comp.windows.x               998
rec.motorcycles              997
comp.graphics                997
comp.sys.ibm.pc.hardware     997
soc.religion.christian       997
comp.sys.mac.hardware        996
comp.os.ms-windows.misc      992
misc.forsale                 991
Name: count, dtype: int64


In [6]:
df.shape

(19955, 4)

In [7]:
df.head()

,file_id,label,text,original_length
0,75895,talk.politics.mideast,in article apr ncsu edu hernlem chess ncsu edu...,1744
1,76248,talk.politics.mideast,ab z virginia edu andi beyer writes uh oh the ...,1602
2,76277,talk.politics.mideast,andrew varvel writes ah c mon give the guy thr...,1810
3,76045,talk.politics.mideast,in article apr thunder mcrcim mcgill edu hasan...,2544
4,77197,talk.politics.mideast,srinivas suder writes precisely why not cuba w...,2365


In [8]:
random_texts = df['text'].sample(n=10)

for i, text in enumerate(random_texts, 1):
    print(f"========== RANDOM DOCUMENT {i} ==========")
    print(text + "...\n")

========== RANDOM DOCUMENT 1 ==========
it s my understanding that koresh didn t want to come out alive rather to fulfill his own interpretations of some prophecies more interesting was discussions on the news about subterannean tunnels under the complex wonder if he was planning on hiding out a few days then re emerging resurrected sounds a bit like the book passover plot to me hmmmmm seth rosenthal disclaimer all opinions are my own not my employers...

========== RANDOM DOCUMENT 2 ==========
in article apr aurora alaska edu nsmca aurora alaska edu writes if you re doing large scale satellite servicing being able to do it in a pressurized hangar makes considerable sense the question is whether anyone is going to be doing large scale satellite servicing in the near future to the point of justifying development of such a thing you d almost certainly use air given that you have to pressurize with something safety considerations strongly suggest making it breathable even if the servicing

### 3. Text Pre-Processing Stage

In [31]:
import nltk
import ssl
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm

In [32]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')       # tokenizer
nltk.download('stopwords')   # stop words list
nltk.download('wordnet')     # lemmatization dictionary
nltk.download('omw-1.4')     # open multilingual wordnet (required by newer NLTK versions)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/manojith/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/manojith/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/manojith/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/manojith/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [33]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [34]:
custom_stop_words = {'write', 'article', 'post', 'know', 'like', 'just', 'say', 'think', 'use', 'make', 'good', 'time', 'people'}
stop_words.update(custom_stop_words)

In [35]:
def nltk_process_text(text):
    tokens = word_tokenize(text) # tokenize the text (splits string into list of words)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(cleaned_tokens)

In [36]:
tqdm.pandas(desc="NLTK Tokenizing, Lemmatizing & Stop Word Removal")

print("Starting NLTK pre-processing...")
df['processed_text'] = df['text'].progress_apply(nltk_process_text)

print("Pre-processing complete! Check the first few rows:")
display(df[['label', 'processed_text']].head())

Starting NLTK pre-processing...


NLTK Tokenizing, Lemmatizing & Stop Word Removal: 100%|██████████| 19955/19955 [00:06<00:00, 2854.06it/s]

Pre-processing complete! Check the first few rows:


,label,processed_text
0,talk.politics.mideast,apr ncsu edu hernlem chess ncsu edu brad hernl...
1,talk.politics.mideast,virginia edu andi beyer writes first sign argu...
2,talk.politics.mideast,andrew varvel writes mon give guy three day se...
3,talk.politics.mideast,apr thunder mcrcim mcgill edu hasan mcrcim mcg...
4,talk.politics.mideast,srinivas suder writes precisely cuba hatians r...


In [38]:
df[['file_id', 'original_length', 'text', 'processed_text', 'label']].head()

,file_id,original_length,text,processed_text,label
0,75895,1744,in article apr ncsu edu hernlem chess ncsu edu...,apr ncsu edu hernlem chess ncsu edu brad hernl...,talk.politics.mideast
1,76248,1602,ab z virginia edu andi beyer writes uh oh the ...,virginia edu andi beyer writes first sign argu...,talk.politics.mideast
2,76277,1810,andrew varvel writes ah c mon give the guy thr...,andrew varvel writes mon give guy three day se...,talk.politics.mideast
3,76045,2544,in article apr thunder mcrcim mcgill edu hasan...,apr thunder mcrcim mcgill edu hasan mcrcim mcg...,talk.politics.mideast
4,77197,2365,srinivas suder writes precisely why not cuba w...,srinivas suder writes precisely cuba hatians r...,talk.politics.mideast
